# Demo: Whisper Inference with Gradio UI

**목적**: 파인튜닝된 Whisper 모델의 추론 결과를 시각적으로 확인하기 위한 Gradio 기반 데모 UI를 구축합니다.

**내용**: 오디오 파일 업로드 → Whisper 추론 → 텍스트 출력의 단방향 파이프라인을 Gradio 인터페이스로 래핑.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers

In [3]:
!pip install torch

In [4]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.2/320.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.2/168.2 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [5]:
_BASE_DIR = '/content/drive/MyDrive/JBNU/2024-2/Capstone/Whisper/'
_DATASETS_DIR = _BASE_DIR + 'datasets/'
_TRANSCRIPT_DIR = _DATASETS_DIR + 'transcript/'

In [6]:
# 저장된 체크포인트 경로 (예: checkpoint-1000)
model_path = _BASE_DIR + "Telos_LLM_early_stop"
processor_path = _BASE_DIR + "Telos_LLM_Processor_early_stop"

In [7]:
from transformers import WhisperForConditionalGeneration
# 모델 불러오기
model = WhisperForConditionalGeneration.from_pretrained(model_path)

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/JBNU/2024-2/Capstone/Whisper/Telos_LLM_early_stop'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
from transformers import WhisperProcessor

# 프로세서 불러오기
processor = WhisperProcessor.from_pretrained(processor_path)

In [ ]:
model.eval()

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 384)
      (layers): ModuleList(
        (0-3): 4 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=384, out_features=384, bias=False)
            (v_proj): Linear(in_features=384, out_features=384, bias=True)
            (q_proj): Linear(in_features=384, out_features=384, bias=True)
            (out_proj): Linear(in_features=384, out_features=384, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          

In [ ]:
# Vocabulary 추출
vocab = processor.tokenizer.get_vocab()

In [ ]:
import json
import pickle

# Vocabulary를 bin 파일로 저장
with open(_BASE_DIR + 'vocab.bin', "wb") as f:
    pickle.dump(vocab, f)


In [ ]:
import gradio as gr
import torch
import librosa
from transformers import WhisperForConditionalGeneration
from transformers import WhisperProcessor


# 저장된 체크포인트 경로 (예: checkpoint-1000)
model_path = _BASE_DIR + "Telos_LLM_early_stop"
processor_path = _BASE_DIR + "Telos_LLM_Processor_early_stop"

# 모델 불러오기
model = WhisperForConditionalGeneration.from_pretrained(model_path)
# 프로세서 불러오기
processor = WhisperProcessor.from_pretrained(processor_path)

model.eval()

# 오디오 처리 및 추론 함수
def transcribe(audio_file):
    # 오디오 파일 로드
    audio_input, _ = librosa.load(audio_file, sr=16000)

    # 입력 데이터 처리
    input_features = processor(audio_input, sampling_rate=16000, return_tensors="pt").input_features

    # 추론 수행
    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    # 생성된 텍스트 디코딩
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

# Gradio 인터페이스 정의
interface = gr.Interface(
    fn=transcribe,  # 추론 함수 연결
    inputs=gr.Audio(type="filepath"),  # 오디오 파일 업로드 입력
    outputs="text",  # 텍스트 출력
    title="Whisper Speech-to-Text",
    description="Upload an audio file, and the Whisper model will transcribe it to text."
)

# Gradio 앱 실행
interface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0197bb22e570a9158.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import torch
import librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

puppy_names = ['강아지', '푸들', '진돗개', '말티즈', '시바견', '비글', '불독', '코카스패니얼', '차우차우', '보더콜리']
cat_names = ['고양이', '러시안블루', '페르시안', '샴고양이', '먼치킨', '스코티시폴드', '노르웨이숲', '뱅갈', '터키시앙고라', '메인쿤']

# 모델과 프로세서 불러오기
checkpoint_path = "kresnik/wav2vec2-large-xlsr-korean"  # 저장된 체크포인트 경로
model = Wav2Vec2ForCTC.from_pretrained(checkpoint_path)
processor = Wav2Vec2Processor.from_pretrained(checkpoint_path)

# 모델 평가 모드로 설정
model.eval()

# 오디오 처리 및 추론 함수
def transcribe(audio_file):

    audio_input, _ = librosa.load(audio_file, sr=16000)

    # 오디오 파일 로드
    inputs = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding="longest")
    input_values = inputs.input_values

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)

    for name in puppy_names:
        if name in transcription:
            return '강아지'

    for name in cat_names:
        if name in transcription:
            return '고양이'

    return transcription

# Gradio 인터페이스 정의
interface = gr.Interface(
    fn=transcribe,  # 추론 함수 연결
    inputs=gr.Audio(type="filepath"),  # 오디오 파일 업로드 입력
    outputs="text",  # 텍스트 출력
    title="Whisper Speech-to-Text",
    description="Upload an audio file, and the Whisper model will transcribe it to text."
)

# Gradio 앱 실행
interface.launch()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.31k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/18.2k [00:00<?, ?B/s]

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ecdf4e939fcb2105ff.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.7.0.post2-cp310-cp310-linux_x86_64.whl size=183291101 sha256=16a849d51b95cf8e47a6e6cd36826e9ffbbc068a8546e7e3501a598bd70905a6
  Stored in directory: /root/.cache/pip/wheels/bf/e3/ed/5e845387d52f2debd1bafb847bf3d774d3f0a3c8e31b1dc948
Successfully built flash-attn


In [8]:
!pip install asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 9.5 MB/s eta 0:00:00


In [12]:
import gradio as gr
import torch
import librosa
import time
from transformers import WhisperForConditionalGeneration
from transformers import WhisperProcessor, AutomaticSpeechRecognitionPipeline
from torch.cuda.amp import autocast
import asyncio

model_name = 'elmenwol/whisper-small_aihub_child'

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 모델 불러오기
model = WhisperForConditionalGeneration.from_pretrained(model_name)
# model = model.half()
# 프로세서 불러오기
processor = WhisperProcessor.from_pretrained(model_name)

model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language='ko', task='transcribe')
pipeline = AutomaticSpeechRecognitionPipeline(
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    generate_kwargs={"forced_decoder_ids": forced_decoder_ids},
    torch_dtype=torch_dtype,
    device=device
)



# 오디오 처리 및 추론 함수
async def transcribe(audio_file):
    start_time = time.time()
    # 생성된 텍스트 디코딩
    with torch.amp.autocast('cuda'):
        loop = asyncio.get_event_loop()
        transcript = await loop.run_in_executor(
            None,
            lambda: pipeline(audio_file, batch_size=8)["text"]
        )
    end_time = time.time()
    print(f"Inference time: {end_time - start_time} seconds")
    return transcript

# Gradio 인터페이스 정의
interface = gr.Interface(
    fn=transcribe,  # 추론 함수 연결
    inputs=gr.Audio(type="filepath"),  # 오디오 파일 업로드 입력
    outputs="text",  # 텍스트 출력
    title="Whisper Speech-to-Text",
    description="Upload an audio file, and the Whisper model will transcribe it to text."
)

# Gradio 앱 실행
interface.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2b583b6cc0342faffd.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
ㄹ